# 🚚 自分たちの街で量子配送 — Real City Edition v2

**「全部の道を同時に試す」量子コンピュータで、本物の地図の上に最短ルートを描く Colab です。**

このノートの主役は2つの問い:

1. **古典コンピュータと量子(QAOA)って、結局なにが違うの?** → このノートでは、配送先を増やしたとき古典がどれだけ苦しむかを実際にストップウォッチで測り、量子の出番がどこで来るのかを見せます。
2. **量子って本当に役に立つの?** → 「直線距離の最短」だけじゃなく、**朝ラッシュの渋滞**を入れて、現実の配送業者と同じ「時間がいちばん短いルート」を解きます。

## 進め方
1. **② チーム情報** と **③ 配送先** を自分の街に書き換える(or 京都6選のまま)
2. メニュー: **ランタイム → すべて実行**
3. 3〜6 分待つ
4. 最後に出る **「距離ルート vs 渋滞回避ルート」の地図**をスクショして成果物に貼る

**所要時間**: 約 6 分 / **前提知識**: 不要

---

## ① 必要なライブラリをインストール

量子計算の `qiskit`、地図の `folium`、住所→座標変換の `geopy` などを入れます。1〜2 分かかります。

In [ ]:
!pip install -q qiskit==1.2.4 qiskit-aer==0.15.1 folium geopy numpy pandas scipy matplotlib japanize-matplotlib
print('✅ install 完了')


## ② チーム情報

右側のフォームで自由に書き換えできます。

In [ ]:
TEAM_NAME = "チーム和風"   #@param {type:"string"}
CITY_NAME = "京都"        #@param {type:"string"}

# ── 出発時刻：渋滞の重さが変わります ──
# "朝ラッシュ" = 幹線が渋滞、 "深夜" = すいすい
DEPART_TIME = "朝ラッシュ(8時)"  #@param ["朝ラッシュ(8時)", "昼(13時)", "深夜(2時)"]

print(f'Team : {TEAM_NAME}')
print(f'City : {CITY_NAME}')
print(f'出発 : {DEPART_TIME}')

## ③ 配送先を入れる(6〜9個)

- 例:「京都駅, 京都市」「清水寺, 京都市」「東京駅, 東京都」など
- 「市」「区」など**場所を絞るキーワード**を入れると見つかりやすいです
- **7〜9 個目は空欄にすると無視されます**。配送先を増やすほど古典は苦しくなり、量子の出番が見えてきます(後の ⑦ でその様子を測ります)
- デフォルトは **京都6選** + オプションで嵐山・宇治・伏見

In [ ]:
PLACE_1 = "京都駅, 京都市"      #@param {type:"string"}
PLACE_2 = "清水寺, 京都市"      #@param {type:"string"}
PLACE_3 = "金閣寺, 京都市"      #@param {type:"string"}
PLACE_4 = "銀閣寺, 京都市"      #@param {type:"string"}
PLACE_5 = "嵐山, 京都市"        #@param {type:"string"}
PLACE_6 = "二条城, 京都市"      #@param {type:"string"}
PLACE_7 = "伏見稲荷大社, 京都市"  #@param {type:"string"}
PLACE_8 = ""                  #@param {type:"string"}
PLACE_9 = ""                  #@param {type:"string"}

_raw = [PLACE_1, PLACE_2, PLACE_3, PLACE_4, PLACE_5, PLACE_6, PLACE_7, PLACE_8, PLACE_9]
places = [p.strip() for p in _raw if p.strip()]

assert 4 <= len(places) <= 9, "配送先は 4～9 個にしてください"

print(f"配送先 {len(places)} 個:")
for i, p in enumerate(places, 1):
    print(f"  {i}. {p}")

## ④ 住所を地図の座標に変換 (geopy / Nominatim)

入れた場所の名前を、地球上の位置(緯度・経度)に変換します。Nominatim は無料の OpenStreetMap 検索。1件ごとに 1.1 秒待つルールなので、9件で約 10〜15 秒。

In [ ]:
import time
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent='quantum-hackathon-2026-v2', timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

coords = []
for i, name in enumerate(places, 1):
    loc = geocode(name)
    if loc is None:
        raise RuntimeError(
            f'見つかりません: {name}。「市」「区」を入れてもう一度試してください'
        )
    coords.append({'name': name, 'lat': loc.latitude, 'lon': loc.longitude})
    print(f'{i}. {name}: ({loc.latitude:.4f}, {loc.longitude:.4f})')

# 倉庫 = 配送先の重心
depot_lat = float(np.mean([c['lat'] for c in coords]))
depot_lon = float(np.mean([c['lon'] for c in coords]))
N_DELIV = len(coords)

print(f'\n倉庫 ({N_DELIV}点の重心): ({depot_lat:.4f}, {depot_lon:.4f})')

## ⑤ 地図プレビュー

緑のピンが配送先、赤いピンが倉庫。位置がおかしかったら ③ に戻って住所を直してください。

In [ ]:
import folium

preview_map = folium.Map(location=[depot_lat, depot_lon], zoom_start=12, tiles='OpenStreetMap')

folium.Marker(
    [depot_lat, depot_lon], popup=f'倉庫 ({CITY_NAME})',
    icon=folium.Icon(color='red', icon='home', prefix='fa')
).add_to(preview_map)

for i, c in enumerate(coords, 1):
    folium.Marker(
        [c['lat'], c['lon']], popup=f"#{i}: {c['name']}",
        icon=folium.Icon(color='green', icon='box', prefix='fa')
    ).add_to(preview_map)

preview_map

## ⑥ 距離マトリックス + 🚦 渋滞マップ

ここが v2 の心臓部です。2種類の「コスト」を作ります。

**(A) 直線距離 (km)** — 地球の丸みを考えた鳥の視点での距離。Haversine 公式で計算。

**(B) 渋滞込みの所要時間** — 現実の配送はこっちが大事。各区間に「混みやすさ」を持たせ、出発時刻が朝ラッシュなら混む区間を **最大1.8倍** 重くします。

> 🧒 **たとえ話**: 同じ「500m先のコンビニ」でも、ガラ空きの夜道なら3分、人混みの朝の駅前なら10分かかりますよね。距離は同じでも「かかる時間」は状況で変わる。配送業者が本当に減らしたいのは km ではなく**時間**です。だから量子に解かせる問題も「時間が最短のルート」にします。

混みやすさは街ごとに毎回同じになるよう固定の乱数で決めています(再現性のため)。

In [ ]:
import numpy as np
import pandas as pd

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# 0 = 倉庫, 1..N = 配送先
all_points = [{"name": "倉庫", "lat": depot_lat, "lon": depot_lon}] + coords
n_points = len(all_points)

# === (A) 直線距離行列 ===
dist = np.zeros((n_points, n_points))
for i in range(n_points):
    for j in range(i + 1, n_points):
        d = haversine_km(all_points[i]["lat"], all_points[i]["lon"],
                         all_points[j]["lat"], all_points[j]["lon"])
        dist[i][j] = dist[j][i] = d

# === 🚦 深滞しやすさ (各区間 0.0～1.0) ===
# 街ごとに毎回同じになるよう、座標をシードにした固定乱数
seed = int(abs(depot_lat * 1000 + depot_lon * 1000)) % (2**32)
rng = np.random.default_rng(seed)
congest = rng.uniform(0, 1, size=(n_points, n_points))
congest = (congest + congest.T) / 2  # 対称に
np.fill_diagonal(congest, 0)

# === 出発時刻 → 渋滞の強さ (rush) ===
rush_map = {"朝ラッシュ(8時)": 1.0, "昼(13時)": 0.45, "深夜(2時)": 0.05}
RUSH = rush_map.get(DEPART_TIME, 1.0)
MAX_MULT = 1.8  # 最も深滞で最大1.8倍

# === (B) 所要時間行列 = 距離 × 渋滞倍率 ===
# 「時速」を仮に 30km/h として分に換算 (km / 30 * 60)
time_mat = np.zeros((n_points, n_points))
for i in range(n_points):
    for j in range(n_points):
        if i == j:
            continue
        mult = 1.0 + (MAX_MULT - 1.0) * congest[i][j] * RUSH
        time_mat[i][j] = dist[i][j] / 30.0 * 60.0 * mult  # 分

labels = [p["name"][:10] for p in all_points]
print(f"出発: {DEPART_TIME}  (渋滞強度 RUSH={RUSH})\n")
print("── (A) 直線距離 (km) ──")
print(pd.DataFrame(dist, index=labels, columns=labels).round(1))
print("\n── (B) 渋滞込み所要時間 (分) ──")
print(pd.DataFrame(time_mat, index=labels, columns=labels).round(1))

## ⑦ 古典コンピュータの限界メーター 📈

> 🧒 **なぜ量子が必要なの?** ここで体感します。
>
> 配送先を全部回る順番は **N!(エヌの階乗)** 通りあります。6個なら 720、でも 10個で 362万、15個で**1兆超え**。これは「配送先が1個増えるたびに、調べる道が10倍以上に膨らむ」ということ。古典コンピュータは1本ずつ調べるので、ある所から**一生かかっても終わらなく**なります。これを「組合せ爆発」と言います。
>
> 量子コンピュータは「全部の道を同時に薄く重ねて試す」ので、この爆発に立ち向かえる可能性がある——それがこのノートで体感したいことです。

下のセルは、配送先を 6→7→8→9→10 と増やしたとき、古典の総当たりが**実際に何秒かかるか**をその場で計測します。あなたのColabのCPUで測るので、数字はマシンによって変わります。

In [ ]:
import time
import math
import numpy as np
from itertools import permutations
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語フォント対応

# === 現在の配送先数で実際のブルートフォースを測定 ===
def brute_seconds(N, mat):
    """N個の配送先を総当たりしてかかった秒と最短コストを返す"""
    idxs = list(range(1, N + 1))
    t0 = time.time()
    best = math.inf
    for p in permutations(idxs):
        c = mat[0][p[0]]
        for a, b in zip(p[:-1], p[1:]):
            c += mat[a][b]
        c += mat[p[-1]][0]
        if c < best:
            best = c
    return time.time() - t0, best

# 測定用: 現在の時間行列を必要なら拡張(足りない分はランダムで埋める)
maxN = 10
base = time_mat.copy()
if base.shape[0] < maxN + 1:
    big = np.random.default_rng(0).uniform(5, 40, size=(maxN + 1, maxN + 1))
    big[:base.shape[0], :base.shape[1]] = base
    np.fill_diagonal(big, 0)
else:
    big = base

Ns, secs, perms_count = [], [], []
print(f"{'配送先':>4}  {'通り数':>12}  {'古典の計算時間':>14}")
print("─" * 38)
for N in range(6, maxN + 1):
    dt, _ = brute_seconds(N, big)
    Ns.append(N); secs.append(dt); perms_count.append(math.factorial(N))
    print(f"{N:>4}  {math.factorial(N):>12,}  {dt:>12.3f}秒")

# === グラフ: 通り数(棒)と実測秒(線) ===
fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.bar([str(n) for n in Ns], perms_count, color="#9aa7b5", alpha=0.7, label="調べる道の数 (N!)")
ax1.set_yscale("log")
ax1.set_ylabel("調べる道の数 (対数目盛り)", color="#5b6770")
ax1.set_xlabel("配送先の数 N")
ax2 = ax1.twinx()
ax2.plot([str(n) for n in Ns], secs, "o-", color="#e8590c", lw=2.5, ms=8, label="古典の実測時間(秒)")
ax2.set_ylabel("古典の計算時間 (秒)", color="#e8590c")
plt.title("配送先が増えると古典は爆発する（組合せ爆発）")
fig.tight_layout()
plt.show()

# メッセージ
n15 = math.factorial(15)
print(f"\nちなみに配送先 15 個なら {n15:,} 通り。")
if secs[-1] > 0:
    rate = perms_count[-1] / secs[-1]
    est15 = n15 / rate
    print(f"今のマシンの速さだと、総当たりに約 {est15/3600:,.1f} 時間かかる計算。")
print("→ ここが「量子の出番」と言われる領域です。")

## ⑧ 今回の問題を確定 — 「距離最短」vs「渋滞回避(時間最短)」

ここでは今あなたが入れた配送先で、2つの答えを古典総当たりで出しておきます(あとで量子の答え合わせに使う)。

- **距離だけ見た最短ルート**(青)
- **渋滞を考えた時間最短ルート**(オレンジ)

この2つが**別のルートになる**のが現実世界のおもしろさ。最後の地図で2本並べて見せます。

In [ ]:
from itertools import permutations
import numpy as np

delivery_indices = list(range(1, N_DELIV + 1))
all_perms = list(permutations(delivery_indices))

def make_cost(mat):
    def tour_cost(perm):
        total = mat[0][perm[0]]
        for a, b in zip(perm[:-1], perm[1:]):
            total += mat[a][b]
        total += mat[perm[-1]][0]
        return total
    return np.array([tour_cost(p) for p in all_perms])

# 距離コストと時間コストの両方
costs_dist = make_cost(dist)
costs_time = make_cost(time_mat)

best_dist_idx = int(np.argmin(costs_dist))
best_time_idx = int(np.argmin(costs_time))

def route_names(perm):
    return [coords[p - 1]["name"].split(",")[0] for p in perm]

print(f"配送先 {N_DELIV} 個 / 巡回順序 = {len(all_perms):,} 通り\n")

print("🔵 距離最短ルート (古典総当たり)")
print(f"   {costs_dist[best_dist_idx]:.2f} km")
print(f"   倉庫 → {' → '.join(route_names(all_perms[best_dist_idx]))} → 倉庫\n")

print("🟠 渋滞回避(時間最短)ルート (古典総当たり)")
print(f"   {costs_time[best_time_idx]:.1f} 分")
print(f"   倉庫 → {' → '.join(route_names(all_perms[best_time_idx]))} → 倉庫\n")

if best_dist_idx != best_time_idx:
    print("✅ 距離最短と時間最短はちがうルートです！")
    print("   「近い道」と「早い道」は別物 — これが現実の配送。")
else:
    print("ℹ️ 今回はたまたま同じルート。出発時刻を「朝ラッシュ」にすると差が出やすいです。")

# 以降の量子パートは「時間最短」を解く(現実の配送に寄せる)
costs = costs_time
best_brute = best_time_idx

## ⑨ 量子のしくみを「波」で理解する 🌊

ここからが量子の本番。難しい言葉を**全部たとえ話**にします。

### 🌊 重ね合わせ = 「全部のルートを同時に薄く試す」
古典コンピュータは「ルートA を試す → ダメ → ルートB を試す…」と**1本ずつ**。
量子コンピュータは最初の操作(Hadamard)で、**全部のルートを同時に、薄〜く重ねた状態**を作ります。

> 🧒 **たとえ**: プールに同時に何百個も小さな波を立てた状態を想像してください。それぞれの波が「1つのルート候補」。最初はどれも同じ高さ(同じ確率)で、まだ「どれがいいか」決まっていません。これが**重ね合わせ**。

### 🎚️ 位相付け (γ:ガンマ) = 「いいルートの波だけ押す」
次に、**短い(良い)ルートの波だけ**タイミングをずらして押してやります。このつまみが **γ(ガンマ)= 短さの好み**。γ を大きくするほど「良いルートを強くえこひいき」します。

### 🎧 干渉 (β:ベータ) = 「波を重ねて、強調と打ち消しを起こす」
最後に波どうしを混ぜる(Rx回転)。すると——
- **良いルートの波どうしは重なって大きくなる**(山と山が重なる=強め合う)
- **悪いルートの波は互いに打ち消し合って消える**(山と谷でゼロ=弱め合う)

> 🧒 **たとえ**: ノイズキャンセリングのイヤホンと同じ原理です。逆向きの波をぶつけると音が消える。量子はこれを使って「悪いルートを消し、良いルートだけ残す」。これを**干渉**と言います。つまみ **β(ベータ)= 混ぜる強さ**。

### 🔁 reps = 「考え直す回数」
γ→β を何回か繰り返すほど、良いルートへの集中が進みます(やりすぎると逆にぼやけることも)。

最後に**測定**すると、波が大きい=確率が高いルートが取り出されやすい。だから「良いルートが出やすい」というわけです。

## ⑩ 量子のつまみ(手動)

3つのつまみを動かして、量子がどう変わるか見てみましょう。意味は ⑨ の通り:
- **γ(ガンマ)= 短さの好み**: 良いルートをどれだけえこひいきするか
- **β(ベータ)= 混ぜる強さ**: 波をどれだけ干渉させるか
- **reps = 考え直す回数**

次の ⑪ で「つまみを自動で良い値に探させる」のもやります。まずは手動で雰囲気をつかんでください。

In [ ]:
GAMMA = 2.55  #@param {type:"slider", min:0, max:3.14, step:0.05}
BETA  = 0.26  #@param {type:"slider", min:0, max:1.57, step:0.02}
REPS  = 5     #@param {type:"slider", min:1, max:10, step:1}

print(f"短さの好み   γ = {GAMMA}")
print(f"混ぜる強さ   β = {BETA}")
print(f"考え直し回数 = {REPS}")

## ⑪ QAOA 回路を作って実行する関数

⑨ で説明した4ステップ(重ね合わせ → 位相付け → 干渉 → 繰り返し → 測定)をそのままコードにします。あとで何度も呼べるように関数 `run_qaoa(γ, β, reps)` にまとめます。

> 💡 各ルートのコスト(時間)を「位相」に変換して、良いルートほど波が強め合うように設計しています。これが量子に "問題" を教える部分です。

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import DiagonalGate
import numpy as np

num_qubits = int(np.ceil(np.log2(len(all_perms))))
size = 1 << num_qubits
INVALID_PENALTY = 1e9

# コストを 2^q の箱に詰める(余りはペナルティ)
cost_vec = np.full(size, INVALID_PENALTY)
cost_vec[:len(costs)] = costs
valid_min, valid_max = costs.min(), costs.max()
span = max(valid_max - valid_min, 1e-9)

_simulator = AerSimulator()

def build_phases(gamma):
    phases = np.empty(size)
    for i in range(size):
        if cost_vec[i] >= INVALID_PENALTY:
            phases[i] = gamma * 1.2          # 無効な箱は不利に
        else:
            norm = (cost_vec[i] - valid_min) / span
            phases[i] = gamma * (1 - norm)    # 良いルートほど強く押す
    return phases

def run_qaoa(gamma, beta, reps, shots=4096):
    diag = np.exp(-1j * build_phases(gamma))
    qc = QuantumCircuit(num_qubits)
    qc.h(range(num_qubits))                   # 🌊 重ね合わせ
    for _ in range(reps):
        qc.append(DiagonalGate(diag.tolist()), list(range(num_qubits)))  # 🎚️ 位相付け(γ)
        for q in range(num_qubits):
            qc.rx(2 * beta, q)                # 🎧 干渉(β)
    qc.measure_all()
    res = _simulator.run(transpile(qc, _simulator), shots=shots).result()
    freq = np.zeros(size)
    for bs, c in res.get_counts().items():
        freq[int(bs.replace(" ", ""), 2)] += c
    valid = freq[:len(all_perms)]
    valid = valid / max(valid.sum(), 1e-12)
    return valid  # 各ルートの測定確率

def expected_cost(valid_probs):
    """測定確率で重み付けした期待コスト(小さいほど良い)"""
    return float(np.sum(valid_probs * costs))

print(f"✅ {num_qubits} qubits / 状態空間 {size} / ルート {len(all_perms):,} 通り")
print("   run_qaoa(γ, β, reps) で何度でも実行できます。")

## ⑫ つまみを自動でチューニング 🎛️ (これが QAOA らしさ)

本物の QAOA は、人がつまみを回すのではなく **コンピュータ自身が「いちばん良いルートが出やすい γ・β」を探します**。ここではそれを再現します。

> 🧒 **たとえ**: ラジオのチューニングと同じ。雑音(悪いルート)が消えて、聞きたい曲(良いルート)がいちばんクリアに入る周波数を、少しずつダイヤルを回して探すイメージ。

いくつかの γ・β を試して、「期待コスト(出てくるルートの平均の短さ)」がいちばん小さくなる組合せを選びます。手動の値(⑩)と比べてみましょう。

In [ ]:
import numpy as np

# いくつかの γ・β を試して、期待コストが最小の組合せを探す
gamma_grid = np.linspace(0.5, 3.0, 6)
beta_grid  = np.linspace(0.1, 0.6, 5)

best = {"cost": np.inf, "g": None, "b": None}
records = []
for g in gamma_grid:
    for b in beta_grid:
        probs = run_qaoa(g, b, REPS, shots=1024)
        ec = expected_cost(probs)
        records.append((g, b, ec))
        if ec < best["cost"]:
            best.update(cost=ec, g=g, b=b)

AUTO_GAMMA, AUTO_BETA = round(best["g"], 2), round(best["b"], 2)
print(f"🎯 自動探索で見つかったベスト: γ={AUTO_GAMMA}, β={AUTO_BETA}")
print(f"   (試した組合せ: {len(records)} 通り)")
print(f"\n手動(⑧)の期待コスト : {expected_cost(run_qaoa(GAMMA, BETA, REPS, 1024)):.2f}")
print(f"自動探索の期待コスト : {best['cost']:.2f}  (小さいほど良い)")

# ヒートマップで γ×β の良さを可視化
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語フォント対応
grid = np.full((len(gamma_grid), len(beta_grid)), np.nan)
for (g, b, ec) in records:
    gi = int(np.argmin(np.abs(gamma_grid - g)))
    bi = int(np.argmin(np.abs(beta_grid - b)))
    grid[gi, bi] = ec
plt.figure(figsize=(6, 4.5))
im = plt.imshow(grid, origin="lower", aspect="auto", cmap="viridis_r",
                extent=[beta_grid[0], beta_grid[-1], gamma_grid[0], gamma_grid[-1]])
plt.colorbar(im, label="期待コスト(小=良い=明るい)")
plt.scatter([AUTO_BETA], [AUTO_GAMMA], c="red", s=120, marker="*",
            edgecolors="white", label="ベスト")
plt.xlabel("β (混ぜる強さ)"); plt.ylabel("γ (短さの好み)")
plt.title("つまみの良さマップ — 赤い星が量子の見つけたベスト")
plt.legend(); plt.tight_layout(); plt.show()

## ⑬ 量子で実行 — 「全部フラット」から「良いルートに集中」へ 🌊→🎯

自動で見つけたベストなつまみで量子を回します。下の2つのヒストグラムを比べてください:

- **左(つまみOFF: γ=0, β=0)**: ただの重ね合わせ。全ルートがほぼ同じ高さ=まだ何も決まっていない波プールの状態
- **右(つまみ最適)**: 干渉で良いルートだけ波が大きくなり、**短いルートに山ができている**

この「フラット → 集中」こそ、量子が干渉で答えを浮かび上がらせている瞬間です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語フォント対応

# OFF(重ね合わせのみ) vs 最適つまみ
probs_off = run_qaoa(0.0, 0.0, 1, shots=4096)
probs_opt = run_qaoa(AUTO_GAMMA, AUTO_BETA, REPS, shots=4096)

# コスト順に並べる(左が良いルート)
order = np.argsort(costs)
x = np.arange(len(order))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].bar(x, probs_off[order] * 100, color="#9aa7b5")
axes[0].set_title("つまみOFF (γ=0, β=0)\n— 全ルートがほぼ同じ = 重ね合わせ")
axes[0].set_xlabel("ルート(左ほど短い→良い)"); axes[0].set_ylabel("出る確率 %")
axes[1].bar(x, probs_opt[order] * 100, color="#2f9e44")
axes[1].set_title(f"つまみ最適 (γ={AUTO_GAMMA}, β={AUTO_BETA})\n— 短いルートに山ができる = 干渉")
axes[1].set_xlabel("ルート(左ほど短い→良い)")
plt.tight_layout(); plt.show()

# 上位5ルート
rank_indices = np.argsort(-probs_opt)
print("🏆 量子の結果 (上位5)")
for rank, idx in enumerate(rank_indices[:5], 1):
    names = [coords[p - 1]["name"].split(",")[0] for p in all_perms[idx]]
    print(f"  #{rank}: 確率 {probs_opt[idx]*100:5.2f}% / 時間 {costs[idx]:6.1f}分")
    print(f"        倉庫 → {' → '.join(names)} → 倉庫")

# 集中度の数値化
top10_off = probs_off[order[:10]].sum() * 100
top10_opt = probs_opt[order[:10]].sum() * 100
print(f"\n上位10良ルートへの集中度:  OFF {top10_off:.1f}%  →  最適 {top10_opt:.1f}%")
print(f"→ つまみを回すと {top10_opt/max(top10_off,1e-9):.1f} 倍良いルートに集中しました。")

## ⑭ 答え合わせ — 量子は古典と同じ最適にたどり着いた?

量子が選んだ1位ルートと、古典総当たりが見つけた本当の最短(時間)を比べます。

> 📝 **大事な視点**: 配送先が6〜9個のうちは古典でも一瞬で解けるので、ここで「量子が勝つ」わけではありません。ここで確認したいのは **「量子もちゃんと正解(かそれに近い所)にたどり着けている」** こと。本当の勝負は ⑦ で見たように、配送先が十数個を超えて古典が現実的な時間で解けなくなる規模です。

In [ ]:
qiskit_top = int(rank_indices[0])

print(f"古典総当たりの最短(時間) : {costs[best_brute]:.1f} 分")
print(f"量子の1位ルート(時間)   : {costs[qiskit_top]:.1f} 分\n")

if qiskit_top == best_brute:
    print("✅ 一致！ 量子が古典と同じ最適ルートを見つけました。")
else:
    gap = (costs[qiskit_top] - costs[best_brute]) / costs[best_brute] * 100
    print(f"量子の1位は最適より +{gap:.1f}% 。")
    rank_of_best = int(np.where(rank_indices == best_brute)[0][0]) + 1
    print(f"（ただし真の最適は量子の {rank_of_best} 位に出ています—探せてはいる）")
    print("→ REPS を増やす / ⑧ の自動探索を細かくすると改善します。")

## ⑮ 結果を地図に描画 📸 — 「距離最短 vs 渋滞回避」2本のルート

成果物のメインビジュアルです。同じ地図に2本のルートを描きます:

- 🔵 **青い破線** = 距離だけ考えた最短ルート
- 🟠 **オレンジの線** = 量子が解いた「渋滞回避(時間最短)」ルート

線は無料の道路ルーティング(OSRM)で**実際の道路に沿わせます**。もし道路データが取れなかった場合は自動で直線に切り替わります(計算した順番は同じ)。

**この地図をスクショして成果物・ストーリーカードに貼ってください。**


In [ ]:
import folium
import urllib.request
import json as _json

# ── 道なりルートを取る(OSRM無料API)、ダメなら直線にフォールバック ──
def road_polyline(lat1, lon1, lat2, lon2):
    """2点間の道なり折れ線[(lat,lon),...]を返す。取れなければ直線2点。"""
    url = (f"https://router.project-osrm.org/route/v1/driving/"
           f"{lon1},{lat1};{lon2},{lat2}?overview=full&geometries=geojson")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "quantum-hackathon-2026/1.0"})
        with urllib.request.urlopen(req, timeout=10) as r:
            data = _json.load(r)
        gj = data["routes"][0]["geometry"]["coordinates"]  # [lon,lat]
        return [(lat, lon) for lon, lat in gj]
    except Exception:
        return [(lat1, lon1), (lat2, lon2)]  # 直線フォールバック

def route_polyline(perm):
    """訪問順の各区間を道なりでつないだ連続折れ線"""
    seq = [(depot_lat, depot_lon)] + [(coords[p-1]["lat"], coords[p-1]["lon"]) for p in perm] + [(depot_lat, depot_lon)]
    line = []
    used_road = True
    for (la1, lo1), (la2, lo2) in zip(seq[:-1], seq[1:]):
        seg = road_polyline(la1, lo1, la2, lo2)
        if len(seg) <= 2:
            used_road = False
        if line and seg and line[-1] == seg[0]:
            line.extend(seg[1:])
        else:
            line.extend(seg)
    return line, used_road

print("道なりルートを取得中... (取れない場合は自動で直線になります)")
time_line, road_ok = route_polyline(all_perms[qiskit_top])
dist_line, _       = route_polyline(all_perms[best_dist_idx])

if road_ok:
    print("   OK: 道なりのルートを描画します。")
else:
    print("   情報: 道路データが取れなかったため直線で描画します(計算結果は同じ)。")

result_map = folium.Map(location=[depot_lat, depot_lon], zoom_start=12, tiles="OpenStreetMap")

# 文字化け対策: charsetを明示
result_map.get_root().header.add_child(folium.Element('<meta charset="utf-8">'))

folium.Marker(
    [depot_lat, depot_lon], popup=f"倉庫 ({CITY_NAME}) - {TEAM_NAME}",
    icon=folium.Icon(color="red", icon="home", prefix="fa")
).add_to(result_map)

for visit_idx, p in enumerate(all_perms[qiskit_top], 1):
    c = coords[p - 1]
    folium.Marker(
        [c["lat"], c["lon"]], popup=f"#{visit_idx}: {c['name']}", tooltip=f"#{visit_idx}",
        icon=folium.Icon(color="green", icon="box", prefix="fa")
    ).add_to(result_map)

# 距離最短(青・破線・細)
folium.PolyLine(
    dist_line, color="#1c7ed6", weight=4, opacity=0.6, dash_array="8,8",
    popup=f"距離最短: {costs_dist[best_dist_idx]:.2f} km"
).add_to(result_map)

# 渋滞回避(オレンジ・量子・太く)
folium.PolyLine(
    time_line, color="#f76707", weight=6, opacity=0.9,
    popup=f"量子の渋滞回避ルート: {costs[qiskit_top]:.1f}分 (gamma={AUTO_GAMMA}, beta={AUTO_BETA})"
).add_to(result_map)

# 凡例(線記号はHTMLエンティティ・font-family明示で文字化けに強く)
road_label = "道なり" if road_ok else "直線"
legend = f'''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
     background: white; padding: 12px 16px; border-radius: 8px;
     box-shadow: 0 2px 8px rgba(0,0,0,.2); font-size: 13px; line-height: 1.8;
     font-family: sans-serif;">
  <b>{TEAM_NAME} &mdash; {CITY_NAME}</b><br>
  <span style="color:#1c7ed6; font-weight:bold;">&#9473;&#9473;</span> 距離最短 ({costs_dist[best_dist_idx]:.1f} km)<br>
  <span style="color:#f76707; font-weight:bold;">&#9473;&#9473;</span> 渋滞回避 ({costs[qiskit_top]:.0f}分 / {road_label})<br>
  <span style="font-size:11px;color:#666;">配送先 {N_DELIV}個 / {len(all_perms):,}通りを量子で探索</span>
</div>
'''
result_map.get_root().html.add_child(folium.Element(legend))

print("\n下の地図をスクショして成果物に貼ってください。")
# Colabはセル最後の地図オブジェクトをそのままインライン表示する
result_map


## 🎯 まとめ — チームの成果物にする

### この地図で言えること(プレゼントーク)
1. 私たちは **{あなたの街}** で、**朝ラッシュの渋滞**を考えた配送ルートを量子で解いた
2. 🔵 距離最短と 🟠 渋滞回避は **別のルート** だった → 「近い ≠ 早い」が地図で一目瞭然
3. つまみ(γ・β)を量子が**自動でチューニング**し、良いルートに集中させた(⑬のヒストグラム)
4. 配送先が増えると古典は爆発する(⑦のグラフ) → ここに量子の出番がある

### 提出物の作り方
1. **⑮ の地図をスクショ**(2本のルートが見えるように)
2. **⑬ のヒストグラム**(フラット→集中)も貼ると「量子が働いた証拠」になる
3. **⑦ のグラフ**(古典の爆発)で「なぜ量子か」を示す

---

### 🔭 もっと先へ(further な発展案)

すぐできる:
- **出発時刻を変えて再実行**: 朝ラッシュ↔深夜でルートがどう変わるか比較スクショ
- **配送先を9個まで増やす**: ③で住所を足す → ⑦のグラフが伸びる

中級:
- **道路距離(osmnx)**: 直線ではなく本物の道路に沿った距離・時間にする(`pip install osmnx`)
- **時間帯を区間ごとに変える**: 朝は駅前、夕方は住宅街が混む、など現実的な渋滞モデル
- **トラックの容量制約(CVRP)**: 「1台で運べる荷物量」を超えたら2台目に分ける → ルート分割問題に拡張
- **時間指定配送(VRPTW)**: 「この家は10-12時希望」など時間枠を制約に追加

本格:
- **IBM Quantum 実機**で動かす([quantum.ibm.com](https://quantum.ibm.com/))
- **本物のQUBO定式化**: ルートのコストを量子ビット間の相互作用(ZZ項)としてエンコードし、配送先が増えても qubit 数が N² で済むようにする(今回の「全順列を箱に詰める」方式は配送先が増えると箱が足りなくなるため、規模拡大にはこちらが必須)
- **配送以外への応用**: 工場の作業順序、避難経路、ゴミ収集、除雪車のルートなど「巡回×制約」の問題はすべて同じ枠組みで解ける

> 💡 量子が本当に強いのは「制約が多くて、選択肢が爆発する現実問題」。渋滞・時間枠・容量を足すほど、このノートは "おもちゃ" から "社会インフラの試作" に近づきます。